In [18]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchsummary import summary
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torchvision
from PIL import Image

train_data = np.load('Numarul5-train/1_5_train.npz', allow_pickle=True)
x_train, y_train = train_data['x'], train_data['y']
x_train = x_train.astype(np.float32)
print(x_train.shape, x_train.dtype, y_train.shape, y_train.dtype)

test_data = np.load('Numarul5-test/1_5_test.npz', allow_pickle=True)
x_test, y_test = test_data['x'], test_data['y']
x_test = x_test.astype(np.float32)
print(x_test.shape, x_test.dtype, y_test.shape, y_test.dtype)

x_train = torch.as_tensor(data=x_train, dtype=torch.float32)
x_test = torch.as_tensor(data=x_test, dtype=torch.float32)

N_CLASSES = len(np.unique(y_train))
print(f"N_CLASSES: {N_CLASSES}")

mean = torch.mean(input=x_train, dim=0)
std = torch.std(input=x_train, dim=0)

x_train_norm = (x_train - mean) / std
x_test_norm = (x_test - mean) / std

n = x_train.shape[0]
K = (x_train_norm.T @ x_train_norm) / (n - 1)
print(K)

D, V = torch.linalg.eigh(K)
idx  = torch.argsort(D, descending=True)

D = D[idx]
V = V[:, idx]


x_train_PCA = x_train_norm @ V[:, :512]
x_test_PCA  = x_test_norm @ V[:, :512]


(32000, 1024) float32 (32000,) int64
(8000, 1024) float32 (8000,) int64
N_CLASSES: 8
tensor([[1.0000, 0.9910, 0.9670,  ..., 0.5426, 0.5304, 0.5200],
        [0.9910, 1.0000, 0.9891,  ..., 0.5523, 0.5390, 0.5275],
        [0.9670, 0.9891, 1.0000,  ..., 0.5644, 0.5497, 0.5371],
        ...,
        [0.5426, 0.5523, 0.5644,  ..., 1.0000, 0.9853, 0.9555],
        [0.5304, 0.5390, 0.5497,  ..., 0.9853, 1.0000, 0.9880],
        [0.5200, 0.5275, 0.5371,  ..., 0.9555, 0.9880, 1.0000]])


In [19]:
class CustomDataset(Dataset):
    def __init__(self, x, y):
        self.feature_vectors = x
        self.labels = y

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        feature_vector = self.feature_vectors[idx]
        label = self.labels[idx]

        feature_vector = torch.as_tensor(data=feature_vector, dtype=torch.float32)
        label = torch.as_tensor(data=label, dtype=torch.int64)

        return feature_vector, label

train_dataset = CustomDataset(x_train_PCA, y_train)
test_dataset = CustomDataset(x_test_PCA, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [20]:
class Model(nn.Module):
    def __init__(self, in_fleatures, n_classes, act1, act2):
        super().__init__()
        
        self.act1 = act1
        self.act2 = act2

        self.fc1 = nn.Linear(in_features=in_fleatures, out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features=in_fleatures)
        self.fc3 = nn.Linear(in_features=in_fleatures, out_features=128)
        self.fc4 = nn.Linear(in_features=128, out_features=n_classes)


    def forward(self, x):
        x = self.act1(self.fc1(x))
        x = self.act2(self.fc2(x))
        x = self.act1(self.fc3(x))
        x = self.fc4(x)

        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(in_fleatures=512, n_classes=N_CLASSES, act1=nn.functional.relu, act2=nn.functional.leaky_relu).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
epochs = 10

def train(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(-1) == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), correct / total

def test(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = loss_fn(out, y)
            total_loss += loss.item()
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return total_loss / len(loader), correct / total


best = 0
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, loss_fn, device)
    test_loss, test_acc = test(model, test_loader, loss_fn, device)
    print(f"train_loss: {train_loss:.4f} || train_acc: {train_acc:.4f} || test_loss: {test_loss:.4f} || test_acc: {test_acc:.4f}")
    if best < test_acc:
        best = test_acc
        torch.save(model, 'model.pt')

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(-1) == y).sum().item()
            total += y.size(0)
        return correct / total


model = torch.load('model.pt', weights_only=False)
test_acc = evaluate(model, test_loader, device)
print(f'{test_acc:.4f}')



train_loss: 1.0001 || train_acc: 0.6713 || test_loss: 0.7846 || test_acc: 0.7590
train_loss: 0.5360 || train_acc: 0.8344 || test_loss: 0.6931 || test_acc: 0.7924
train_loss: 0.3848 || train_acc: 0.8792 || test_loss: 0.7472 || test_acc: 0.7983
train_loss: 0.3018 || train_acc: 0.9042 || test_loss: 0.6775 || test_acc: 0.8150
train_loss: 0.2391 || train_acc: 0.9236 || test_loss: 0.7142 || test_acc: 0.8169
train_loss: 0.1887 || train_acc: 0.9403 || test_loss: 0.7851 || test_acc: 0.8177
train_loss: 0.1589 || train_acc: 0.9500 || test_loss: 0.8632 || test_acc: 0.8165
train_loss: 0.1374 || train_acc: 0.9564 || test_loss: 0.9223 || test_acc: 0.8141
train_loss: 0.1133 || train_acc: 0.9627 || test_loss: 0.9452 || test_acc: 0.8133
train_loss: 0.0930 || train_acc: 0.9698 || test_loss: 0.9907 || test_acc: 0.8149
0.8177
